# Era of Experience · Experience Lab 1.16.0

Run all four bounded, synthetic learning experiments and verify their evidence. No API key, GPU, provider, model download or pip install is used. Source retrieval requires network access once; subsequent cells run locally.

The original notebook is preserved as `research_notebook_archive.ipynb`. [Project notice](https://github.com/MontrealAI/AGI-Alpha-Agent-v0/blob/main/docs/DISCLAIMER_SNIPPET.md).

SPDX-License-Identifier: Apache-2.0

In [ ]:
import pathlib, subprocess, sys
assert (3, 11) <= sys.version_info[:2] < (3, 14), "Use Python 3.11–3.13"
release = "v1.16.0"
# Reuse a local checkout when present; otherwise retrieve the exact release.
root = next((p for p in (pathlib.Path.cwd(), *pathlib.Path.cwd().parents)
             if (p / "alpha_factory_v1/demos/era_of_experience/lab.py").is_file()), None)
if root is None:
    root = pathlib.Path.cwd() / ("AGI-Alpha-Agent-" + release)
    if not root.exists():
        subprocess.run(["git", "clone", "--depth", "1", "--branch", release,
                        "https://github.com/MontrealAI/AGI-Alpha-Agent-v0.git", str(root)], check=True, timeout=180)
    assert subprocess.check_output(["git", "-C", str(root), "describe", "--tags", "--exact-match"], text=True).strip() == release
sys.path.insert(0, str(root))
print("Source:", root)
print("Commit:", subprocess.check_output(["git", "-C", str(root), "rev-parse", "HEAD"], text=True).strip())


In [ ]:
from alpha_factory_v1.demos.era_of_experience.lab import read_json, evaluate, verify, write_bundle, brief
cases = read_json(root / "alpha_factory_v1/demos/era_of_experience/scenarios.json")
reports = [evaluate(case) for case in cases]
for report in reports:
    assert verify(report) == report
    print(report["input"]["title"], report["result"]["status"], report["sha256"])


## Inspect a result

The candidate is frozen before evaluation. A separate seed stream is used for each test suite; neither changes learning memory. A passing result is a request for independent review, not deployment approval.

In [ ]:
print(brief(reports[0]))
trap = reports[2]
assert trap["result"]["status"] == "HOLD_BASELINE"
print("Reward trap:", [gate["label"] for gate in trap["result"]["gates"] if not gate["passed"]])


In [ ]:
output = pathlib.Path.cwd() / "experience-notebook-runs"
folders = [write_bundle(report, output) for report in reports]
for folder in folders:
    verify(read_json(folder / "run.json"))
print("Verified evidence folders:")
for folder in folders:
    print(folder)


## Make one change

Use a new seed before running a genuinely independent follow-up. Do not repeatedly select seeds based on which one passes. Edit any scenario below to study exploration, memory, reward design or environment shift.

In [ ]:
from copy import deepcopy
changed = deepcopy(cases[0])
changed["training"]["memoryWindow"] = 8
changed["training"]["explorationBps"] = 3000
changed_report = evaluate(changed)
print(brief(changed_report))
write_bundle(changed_report, output)


The [browser lab](https://montrealai.github.io/AGI-Alpha-Agent-v0/era_of_experience/) can import any exported `run.json`. For a local interface from a terminal: `python -m alpha_factory_v1.demos.era_of_experience --serve`. The notebook does not start a public tunnel. Original research and diagrams remain in `RESEARCH_ARCHIVE.md`.